In [1]:
output_path = "/opt/consumer-output-delta-lake/delta-trades-stream-5s-table"
checkpoint_path = "/opt/consumer-output-delta-lake/checkpoints/delta-trades-stream-5s-table"
ram_limit = 4

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

## 1. Create spark app and read from kafka stream

In [2]:
# --------------------------------------------------
# 1. Create Spark session WITH DELTA LAKE SUPPORT
# --------------------------------------------------
spark = (
    SparkSession.builder
    .appName("KafkaStreamingToConsole")
    .config("spark.ui.port", "4042")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", f"{ram_limit}g")  # or 6g if you have RAM
    .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# --------------------------------------------------
# 2. Read from Kafka (STREAMING)
# --------------------------------------------------
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "trades.raw")
    .option("startingOffsets", "latest")
    .load()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/30 04:52:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Data transformation

In [ ]:
# --------------------------------------------------
# 3-7. Transformations (unchanged)
# --------------------------------------------------
df2 = df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

value_schema = StructType([
    StructField("exchange", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("trade_id", LongType(), True),
    StructField("side", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("size", DoubleType(), True),
    StructField("exchange_time", StringType(), True),
    StructField("ingest_time", StringType(), True),
])

df_parsed = df2.withColumn("value_json", from_json(col("value"), value_schema))

df_flat = df_parsed.select(
    col("offset"),
    col("key"),
    col("value_json.symbol").alias("symbol"),
    col("value_json.trade_id").alias("trade_id"),
    col("value_json.side").alias("side"),
    col("value_json.price").alias("price"),
    col("value_json.size").alias("size"),
    col("value_json.exchange_time").alias("exchange_time"),
    col("value_json.ingest_time").alias("ingest_time"),
)

df_final = (
    df_flat
    .withColumn("offset", col("offset").cast("long"))
    .withColumn("trade_id", col("trade_id").cast("long"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("size", col("size").cast("double"))

    .withColumn("exchange_time_ts", from_utc_timestamp(to_timestamp("exchange_time"), "Asia/Bangkok").cast("timestamp"))
    .withColumn("ingest_time_ts", from_utc_timestamp(to_timestamp("ingest_time"), "Asia/Bangkok").cast("timestamp"))
    .withColumn("ingest_time_minute", date_trunc("minute", col("ingest_time_ts")).cast("timestamp"))

    .withColumn("exchange_second", second("exchange_time_ts").cast("int"))
    .withColumn("exchange_minute", minute("exchange_time_ts").cast("int"))
    .withColumn("exchange_hour", hour("exchange_time_ts").cast("int"))
    .withColumn("exchange_day", dayofmonth("exchange_time_ts").cast("int"))
    .withColumn("exchange_month", month("exchange_time_ts").cast("int"))

    .withColumn("hour_id", date_format("ingest_time_ts", "yyyyMMddHH").cast("string"))
)

# --------------------------------------------------
# 8. DISPLAY STREAMING RESULTS TO CONSOLE ✅
# --------------------------------------------------
query = (
    df_final.writeStream
    .outputMode("append")
    .format("console")          # ← Prints to stdout
    .option("truncate", False)  # ← Show full column content
    .option("numRows", 20)      # ← Show up to 20 rows per batch
    .start()
)

print("Streaming started. Watching Kafka topic 'trades.raw'...")
query.awaitTermination()  # Keeps the stream alive

25/12/30 04:53:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-66ad3323-c5fb-4ee5-93bd-69d8ebded678. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/30 04:53:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming started. Watching Kafka topic 'trades.raw'...
-------------------------------------------
Batch: 0
-------------------------------------------
+------+---+------+--------+----+-----+----+-------------+-----------+----------------+--------------+------------------+---------------+---------------+-------------+------------+--------------+-------+
|offset|key|symbol|trade_id|side|price|size|exchange_time|ingest_time|exchange_time_ts|ingest_time_ts|ingest_time_minute|exchange_second|exchange_minute|exchange_hour|exchange_day|exchange_month|hour_id|
+------+---+------+--------+----+-----+----+-------------+-----------+----------------+--------------+------------------+---------------+---------------+-------------+------------+--------------+-------+
+------+---+------+--------+----+-----+----+-------------+-----------+----------------+--------------+------------------+---------------+---------------+-------------+------------+--------------+-------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-------+-------+---------+----+--------+----------+---------------------------+--------------------------------+--------------------------+--------------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
|offset |key    |symbol |trade_id |side|price   |size      |exchange_time              |ingest_time                     |exchange_time_ts          |ingest_time_ts            |ingest_time_minute |exchange_second|exchange_minute|exchange_hour|exchange_day|exchange_month|hour_id   |
+-------+-------+-------+---------+----+--------+----------+---------------------------+--------------------------------+--------------------------+--------------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
|511687 |XRP-USD|XRP-USD|202899024|buy |1.8599  |2500.0    |

## 3. Write to delta stream table with checkpoint

In [ ]:
# --------------------------------------------------
# 4. Write stream to Delta (CHECKPOINT IS CRITICAL)
# --------------------------------------------------
query = (
    df_final.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(processingTime='5 seconds')  # ← ADD THIS LINE
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("hour_id")          # ✅ PARTITION HERE
    .start(output_path)
)

print("Streaming started → writing to Delta")
query.awaitTermination()

25/12/29 09:24:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming started → writing to Delta


25/12/29 09:24:54 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 7721 milliseconds
25/12/29 09:24:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/12/29 09:24:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5677 milliseconds
25/12/29 09:25:05 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5122 milliseconds
25/12/29 09:25:11 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5913 milliseconds
25/12/29 09:33:15 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5335 milliseconds
25/12/29 09:34:05 WARN ProcessingTimeExecutor: Current b